# Interactive kNN graph coarsening

Choose one of the 2D synthetic datasets, adjust the base kNN neighbor count, and move the coarsening slider to change the number of k-means representatives. The left view shows the observation-level kNN graph with light edges; the right view shows the reduced graph whose nodes are k-means centroids and whose edges are induced by the selected base kNN graph.

In [1]:
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.collections import LineCollection
from sklearn.cluster import KMeans
from sklearn.neighbors import kneighbors_graph

# Make the notebook work when launched from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'topological_graph_embedding').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from topological_graph_embedding.datasets import generate_synthetic_datasets

N_SAMPLES = 500
DEFAULT_KNN_NEIGHBORS = 6
KNN_MIN = 2
KNN_MAX = 12
RANDOM_STATE = 7

datasets = generate_synthetic_datasets(
    n=N_SAMPLES,
    noise=0.045,
    random_state=RANDOM_STATE,
)
dataset_names = list(datasets)


def symmetric_knn_graph(points, n_neighbors=DEFAULT_KNN_NEIGHBORS):
    """Return the undirected, distance-weighted observation kNN graph."""
    directed = kneighbors_graph(
        points,
        n_neighbors=n_neighbors,
        mode='distance',
        include_self=False,
    )
    return directed.maximum(directed.T).tocsr()


knn_graph_cache = {}


def get_knn_graph(dataset_name, n_neighbors):
    key = (dataset_name, int(n_neighbors))
    if key not in knn_graph_cache:
        knn_graph_cache[key] = symmetric_knn_graph(
            datasets[dataset_name], n_neighbors=int(n_neighbors)
        )
    return knn_graph_cache[key]

print(f'{len(datasets)} datasets ready; each has {N_SAMPLES} observations and a base k of {DEFAULT_KNN_NEIGHBORS}.')

7 datasets ready; each has 500 observations and 6-NN edges.


In [2]:
def undirected_edges(graph):
    """Extract each symmetric sparse-graph edge once."""
    coo = graph.tocoo()
    keep = coo.row < coo.col
    edges = np.column_stack([coo.row[keep], coo.col[keep]])
    distances = np.asarray(coo.data[keep], dtype=float)
    return edges.astype(int), distances


def coarsen_knn_graph(points, graph, n_centroids):
    """Fit centroids and contract the observation kNN edges between clusters."""
    model = KMeans(
        n_clusters=int(n_centroids),
        n_init=10,
        random_state=RANDOM_STATE,
    )
    labels = model.fit_predict(points)
    centers = model.cluster_centers_
    sizes = np.bincount(labels, minlength=len(centers))

    fine_edges, _ = undirected_edges(graph)
    left = labels[fine_edges[:, 0]]
    right = labels[fine_edges[:, 1]]
    crosses_cluster = left != right
    pairs = np.sort(
        np.column_stack([left[crosses_cluster], right[crosses_cluster]]),
        axis=1,
    )
    if len(pairs):
        pair_codes = pairs[:, 0] * len(centers) + pairs[:, 1]
        unique_codes, support = np.unique(pair_codes, return_counts=True)
        coarse_edges = np.column_stack(
            np.unravel_index(unique_codes, (len(centers), len(centers)))
        )
    else:
        coarse_edges = np.empty((0, 2), dtype=int)
        support = np.empty(0, dtype=int)
    return centers, sizes, coarse_edges.astype(int), support.astype(int)


def _limits(points, pad=0.12):
    minimum = points.min(axis=0)
    maximum = points.max(axis=0)
    span = np.maximum(maximum - minimum, 1e-9)
    return (minimum[0] - pad * span[0], maximum[0] + pad * span[0],
            minimum[1] - pad * span[1], maximum[1] + pad * span[1])


def _style_axis(axis, points):
    axis.set_aspect('equal', adjustable='box')
    axis.set_xlim(_limits(points)[0:2])
    axis.set_ylim(_limits(points)[2:4])
    axis.set_xlabel('feature 1')
    axis.set_ylabel('feature 2')
    axis.grid(alpha=0.15, linewidth=0.6)


def render_view(dataset_name, n_centroids, n_neighbors):
    points = datasets[dataset_name]
    graph = get_knn_graph(dataset_name, n_neighbors)
    fine_edges, _ = undirected_edges(graph)
    centers, sizes, coarse_edges, support = coarsen_knn_graph(
        points, graph, n_centroids
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    fine_segments = points[fine_edges]
    axes[0].add_collection(LineCollection(
        fine_segments,
        colors='#b9d7e8',
        linewidths=0.55,
        alpha=0.52,
        zorder=1,
    ))
    axes[0].scatter(
        points[:, 0], points[:, 1],
        s=10, c='#27648a', alpha=0.72, linewidths=0, zorder=2,
    )
    axes[0].set_title(f'{dataset_name}: {n_neighbors}-NN graph')
    _style_axis(axes[0], points)

    # Keep the original cloud faintly visible so the coarsening is easy to follow.
    axes[1].scatter(
        points[:, 0], points[:, 1],
        s=7, c='#d9e6ed', alpha=0.48, linewidths=0, zorder=1,
    )
    if len(coarse_edges):
        coarse_segments = centers[coarse_edges]
        relative_support = support / max(float(support.max()), 1.0)
        axes[1].add_collection(LineCollection(
            coarse_segments,
            colors='#356f8f',
            linewidths=0.8 + 2.2 * relative_support,
            alpha=0.82,
            zorder=2,
        ))
    axes[1].scatter(
        centers[:, 0], centers[:, 1],
        s=24 + 90 * sizes / max(float(sizes.max()), 1.0),
        c='#d95f59', edgecolors='#7f2d2a', linewidths=0.7,
        alpha=0.95, zorder=3,
    )
    axes[1].set_title(
        f'{dataset_name}: {len(centers)} coarsened centroids + edges'
    )
    _style_axis(axes[1], points)

    fig.suptitle(
        'Observation graph to k-means reduced graph',
        fontsize=15,
    )
    display(fig)
    plt.close(fig)


## Explore the coarsening

The reduced edges connect two centroids whenever at least one original kNN edge crossed between their clusters. Thicker edges have more supporting observation-level edges.

In [ ]:
dataset_selector = widgets.Dropdown(
    options=dataset_names,
    value=dataset_names[0],
    description='dataset',
    style={'description_width': 'initial'},
)
centroid_slider = widgets.IntSlider(
    value=32,
    min=8,
    max=96,
    step=4,
    continuous_update=False,
    description='coarsened k-means',
    style={'description_width': 'initial'},
)
knn_slider = widgets.IntSlider(
    value=DEFAULT_KNN_NEIGHBORS,
    min=KNN_MIN,
    max=KNN_MAX,
    step=1,
    continuous_update=False,
    description='base k (NN)',
    style={'description_width': 'initial'},
)

controls = widgets.HBox(
    [dataset_selector, knn_slider, centroid_slider],
    layout=widgets.Layout(display='flex', flex_flow='row wrap', gap='18px'),
)
output = widgets.interactive_output(
    render_view,
    {
        'dataset_name': dataset_selector,
        'n_centroids': centroid_slider,
        'n_neighbors': knn_slider,
    },
)
display(controls, output)

Output()